# MA3632 — Workshop 4: Feature Engineering and Dimensionality Reduction

This workshop accompanies Lecture 4 and closes the Data Preparation block.
Part A covers feature construction. Part B covers the curse of dimensionality
and filter-based feature selection. Part C works through PCA from the SVD,
scree plots, and variance explained. Part D applies PCA inside a full pipeline
and compares performance with and without dimensionality reduction.

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing, load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

rng = np.random.default_rng(seed=0)

# California Housing — used in Parts A and D.
# Falls back to a locally generated synthetic dataset with the same columns
# if the network fetch is unavailable, so the workshop runs offline.
try:
    raw = fetch_california_housing(as_frame=True)
    df_housing = raw.frame.copy()
except Exception as e:
    print(f"California Housing fetch unavailable ({type(e).__name__}); using synthetic fallback.")
    n_synth = 3000
    rng_synth = np.random.default_rng(seed=42)
    med_inc    = rng_synth.gamma(shape=5.0, scale=0.8, size=n_synth)
    house_age  = rng_synth.uniform(1, 52, size=n_synth)
    ave_rooms  = rng_synth.normal(5.5, 1.2, size=n_synth).clip(min=1)
    ave_bedrms = ave_rooms * rng_synth.uniform(0.15, 0.35, size=n_synth)
    population = rng_synth.gamma(shape=3.0, scale=400, size=n_synth)
    ave_occup  = rng_synth.normal(3.0, 0.7, size=n_synth).clip(min=1)
    latitude   = rng_synth.uniform(32.5, 42.0, size=n_synth)
    longitude  = rng_synth.uniform(-124.3, -114.3, size=n_synth)
    med_house_val = (
        0.5 * med_inc + 0.01 * (52 - house_age) - 0.05 * ave_occup
        + rng_synth.normal(0, 0.4, size=n_synth)
    ).clip(min=0.15, max=5.0)
    df_housing = pd.DataFrame({
        "MedInc": med_inc, "HouseAge": house_age, "AveRooms": ave_rooms,
        "AveBedrms": ave_bedrms, "Population": population, "AveOccup": ave_occup,
        "Latitude": latitude, "Longitude": longitude, "MedHouseVal": med_house_val,
    })

X_housing = df_housing.drop(columns="MedHouseVal")
y_housing  = df_housing["MedHouseVal"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X_housing, y_housing, test_size=0.2, random_state=0
)

print("California Housing — training set:", X_tr.shape)

# Wine dataset — used in Parts B and C (higher dimensional, classification).
# Split immediately, before any scaling or filtering, following the same
# train-only rule that Part D makes explicit for PCA.
wine = load_wine(as_frame=True)
X_wine, y_wine = wine.data, wine.target

X_wine_tr, X_wine_te, y_wine_tr, y_wine_te = train_test_split(
    X_wine, y_wine, test_size=0.2, random_state=0, stratify=y_wine
)

print("Wine dataset — training set:      ", X_wine_tr.shape)

---

## Part A — Feature construction

### A1. Ratio and interaction features

In [ ]:
# California Housing already contains several implicit ratio features.
# We construct a few additional ones that encode domain knowledge.

df_fe = X_tr.copy()

# Rooms per household (AveRooms is already this, but AveBedrms/AveRooms is new)
df_fe["bedroom_ratio"] = X_tr["AveBedrms"] / X_tr["AveRooms"]

# Rooms per person: living space per occupant, distinct from AveRooms
# (per household) or AveOccup (people per household) individually
df_fe["rooms_per_person"] = X_tr["AveRooms"] / X_tr["AveOccup"].clip(lower=1)

# Income per occupant proxy
df_fe["income_per_occupant"] = X_tr["MedInc"] / X_tr["AveOccup"].clip(lower=1)

print("Original features:", list(X_tr.columns))
print("\nEngineered features:", ["bedroom_ratio", "rooms_per_person", "income_per_occupant"])
print()
print(df_fe[["bedroom_ratio", "rooms_per_person", "income_per_occupant"]].describe().round(3))

In [ ]:
# Check correlation of engineered features with the target
y_tr_aligned = y_tr.loc[df_fe.index]

corr_orig = X_tr.corrwith(y_tr_aligned).abs().sort_values(ascending=False)
corr_new  = df_fe[["bedroom_ratio", "rooms_per_person", "income_per_occupant"]].corrwith(y_tr_aligned).abs()

print("Correlation with target — original features:")
print(corr_orig.round(3).to_string())
print("\nCorrelation with target — engineered features:")
print(corr_new.round(3).to_string())

### A2. Polynomial features

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.metrics import root_mean_squared_error
from sklearn.pipeline import Pipeline

# Compare degree-1 vs degree-2 polynomial expansion on a single feature
feature = "MedInc"
X_single_tr = X_tr[[feature]]
X_single_te = X_te[[feature]]

results = {}
for degree in [1, 2, 3]:
    pipe = Pipeline([
        ("poly",   PolynomialFeatures(degree=degree, include_bias=False)),
        ("scaler", StandardScaler()),
        ("model",  Ridge(alpha=1.0)),
    ])
    pipe.fit(X_single_tr, y_tr)
    pred = pipe.predict(X_single_te)
    rmse = root_mean_squared_error(y_te, pred)
    n_features = pipe.named_steps["poly"].n_output_features_
    results[degree] = {"RMSE": rmse, "n_features": n_features}
    print(f"Degree {degree}: {n_features} feature(s), RMSE = {rmse:.4f}")

In [ ]:
# Visualise the fitted curves
x_grid = np.linspace(X_single_tr[feature].min(), X_single_tr[feature].max(), 200).reshape(-1, 1)
x_grid_df = pd.DataFrame(x_grid, columns=[feature])

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(X_single_tr[feature], y_tr, alpha=0.1, s=8, color="steelblue", label="training data")

colors = ["firebrick", "darkorange", "forestgreen"]
for degree, color in zip([1, 2, 3], colors):
    pipe = Pipeline([
        ("poly",   PolynomialFeatures(degree=degree, include_bias=False)),
        ("scaler", StandardScaler()),
        ("model",  Ridge(alpha=1.0)),
    ])
    pipe.fit(X_single_tr, y_tr)
    ax.plot(x_grid, pipe.predict(x_grid_df), color=color, linewidth=1.5,
            label=f"degree {degree}")

ax.set_xlabel("MedInc")
ax.set_ylabel("MedHouseVal")
ax.set_title("Polynomial regression on MedInc")
ax.legend()
plt.tight_layout()
plt.show()

**Exercise A.** Construct two additional ratio features for the California Housing
dataset that you think are likely to be predictive of house value, and justify each
choice in one sentence. Add them to `df_fe`, fit a Ridge regression on the augmented
training set (after standardising), and compare the test RMSE to a baseline fitted
on the original eight features only.

---

## Part B — Curse of dimensionality and filter selection

### B1. Distance concentration

In [ ]:
# Demonstrate that all pairwise distances become similar as dimension grows
def distance_ratio(p, n=500, seed=1):
    rng_local = np.random.default_rng(seed)
    X = rng_local.uniform(0, 1, size=(n, p))
    # Compute pairwise distances for a random subset of 100 pairs
    idx = rng_local.choice(n, size=(200, 2), replace=False)
    dists = np.array([
        np.linalg.norm(X[i] - X[j]) for i, j in idx
    ])
    return (dists.max() - dists.min()) / dists.min()

dims = [1, 2, 5, 10, 20, 50, 100, 200]
ratios = [distance_ratio(p) for p in dims]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(dims, ratios, marker="o", color="steelblue", linewidth=1.5)
ax.set_xlabel("Dimension p")
ax.set_ylabel("(d_max - d_min) / d_min")
ax.set_title("Distance concentration as dimension increases")
ax.set_xscale("log")
plt.tight_layout()
plt.show()

print("Distance ratio by dimension:")
for p, r in zip(dims, ratios):
    print(f"  p = {p:>4}:  {r:.4f}")

As predicted by the theory in Lecture 4, the ratio of maximum to minimum distance
falls toward zero as $p$ grows: all points become approximately equidistant, and
nearest-neighbour methods lose discriminating power.

### B2. Filter methods

In [ ]:
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif, mutual_info_classif

# Work on the Wine training set only (13 features, 3 classes)
scaler = StandardScaler()
X_wine_scaled = scaler.fit_transform(X_wine_tr)

print("Wine dataset — feature variances (after scaling, should all be ~1):")
variances = X_wine_scaled.var(axis=0)
for name, v in zip(wine.feature_names, variances):
    print(f"  {name:<30} {v:.4f}")

In [ ]:
# Variance threshold on raw (unscaled) wine training data to illustrate the concept
vt = VarianceThreshold(threshold=0.5)
vt.fit(X_wine_tr)
kept = np.array(wine.feature_names)[vt.get_support()]
removed = np.array(wine.feature_names)[~vt.get_support()]
print(f"Features retained (variance > 0.5): {list(kept)}")
print(f"Features removed:                   {list(removed)}")

In [ ]:
# Correlation filter: remove one from each highly correlated pair
corr_matrix = pd.DataFrame(X_wine_tr, columns=wine.feature_names).corr().abs()

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr_matrix, cmap="coolwarm", vmin=0, vmax=1)
ax.set_xticks(range(len(wine.feature_names)))
ax.set_yticks(range(len(wine.feature_names)))
ax.set_xticklabels(wine.feature_names, rotation=45, ha="right", fontsize=7)
ax.set_yticklabels(wine.feature_names, fontsize=7)
plt.colorbar(im, ax=ax)
ax.set_title("Wine dataset (training set) — absolute pairwise correlations")
plt.tight_layout()
plt.show()

# Identify pairs with |r| > 0.75
threshold = 0.75
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_pairs = [(c, r) for c in upper.columns for r in upper.index
                   if pd.notna(upper.loc[r, c]) and upper.loc[r, c] > threshold]
print(f"\nFeature pairs with |r| > {threshold}:")
for a, b in high_corr_pairs:
    print(f"  {a:<30} — {b:<30} r = {corr_matrix.loc[a, b]:.3f}")

In [ ]:
# Univariate selection: F-statistic and mutual information (training set only)
k = 6   # retain top-6 features

sel_f  = SelectKBest(score_func=f_classif, k=k)
sel_mi = SelectKBest(score_func=lambda X, y: mutual_info_classif(X, y, random_state=0), k=k)

sel_f.fit(X_wine_tr, y_wine_tr)
sel_mi.fit(X_wine_tr, y_wine_tr)

features_f  = np.array(wine.feature_names)[sel_f.get_support()]
features_mi = np.array(wine.feature_names)[sel_mi.get_support()]

print(f"Top {k} by F-statistic:       {list(features_f)}")
print(f"Top {k} by mutual information: {list(features_mi)}")
print()

# Are they the same?
overlap = set(features_f) & set(features_mi)
print(f"Features selected by both: {sorted(overlap)}")

**Exercise B.** Apply the three filter steps (variance threshold, correlation filter,
F-statistic selection) sequentially to the Wine training set (`X_wine_tr`, `y_wine_tr`)
to produce a reduced feature set. Fit a $k$-nearest neighbour classifier ($k=5$) on
the full feature set and on the reduced set, in each case fitting on the training set
and evaluating on the held-out test set (`X_wine_te`, `y_wine_te`). Compare test
accuracy. Does filter selection help here, and why might that be the case given the
curse of dimensionality argument from B1?

---

## Part C — Principal Component Analysis

### C1. PCA from the SVD

In [ ]:
# Centre and scale the Wine training data, then compute SVD manually
X_c = X_wine_scaled   # already centred and scaled by StandardScaler above (training set)

U, d, Vt = np.linalg.svd(X_c, full_matrices=False)
V = Vt.T              # columns of V are the PC directions

# Eigenvalues of the covariance matrix
n = X_c.shape[0]
lambdas = d**2 / (n - 1)

print("Singular values d_k:")
print(np.round(d, 4))
print("\nEigenvalues lambda_k = d_k^2 / (n-1):")
print(np.round(lambdas, 4))
print(f"\nTrace of S (total variance): {lambdas.sum():.4f}")
print(f"Expected (p features, unit variance): {X_wine.shape[1]}")

In [ ]:
# Verify uncorrelatedness of principal components
Z = X_c @ V   # principal components, shape (n, p)
cov_Z = np.round(Z.T @ Z / (n - 1), 6)

print("Sample covariance matrix of principal components (should be diagonal):")
print(pd.DataFrame(cov_Z).round(4).to_string())

### C1b. A 2D geometric warm-up

Before working in 13 dimensions, it helps to see what a principal component actually
looks like. We construct a small synthetic dataset with the same covariance structure
as the worked example in Lecture 4, Section 3.2 ($\lambda_1 = 1.8$, $\lambda_2 = 0.2$),
and plot the data together with the eigenvectors of its covariance matrix.

In [ ]:
# Synthetic 2D data matching the covariance structure of Lecture 4, Section 3.2:
# S = [[1, 0.8], [0.8, 1]], eigenvalues 1.8 and 0.2
rng_2d = np.random.default_rng(seed=7)
S_target = np.array([[1.0, 0.8], [0.8, 1.0]])
L_chol = np.linalg.cholesky(S_target)
n_2d = 300
Z_2d = rng_2d.standard_normal((n_2d, 2))
X_2d = Z_2d @ L_chol.T   # correlated samples with (approximately) the target covariance
X_2d = X_2d - X_2d.mean(axis=0)   # centre

S_2d = (X_2d.T @ X_2d) / (n_2d - 1)
eigvals_2d, eigvecs_2d = np.linalg.eigh(S_2d)   # ascending order
order = np.argsort(eigvals_2d)[::-1]
eigvals_2d = eigvals_2d[order]
eigvecs_2d = eigvecs_2d[:, order]

print("Sample covariance matrix:")
print(np.round(S_2d, 3))
print(f"\nEigenvalues: {np.round(eigvals_2d, 3)}")
print(f"Variance explained by PC1: {eigvals_2d[0] / eigvals_2d.sum() * 100:.1f}%")

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(X_2d[:, 0], X_2d[:, 1], alpha=0.35, s=15, color="steelblue")

origin = np.zeros(2)
for k, (color, label) in enumerate(zip(["firebrick", "forestgreen"], ["PC1", "PC2"])):
    vec = eigvecs_2d[:, k] * np.sqrt(eigvals_2d[k]) * 2   # scale for visibility
    ax.annotate("", xy=vec, xytext=origin,
                arrowprops=dict(arrowstyle="->", color=color, linewidth=2))
    ax.text(*(vec * 1.1), label, color=color, fontsize=11, fontweight="bold")

ax.set_xlabel("$X_1$")
ax.set_ylabel("$X_2$")
ax.set_title("Principal components as the axes of the covariance ellipse")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

### C1c. Cross-checking against the covariance eigendecomposition

Theorem 3.1 of Lecture 4 characterises the principal component directions as the
eigenvectors of $\mathbf{S}$ directly, obtained here via the SVD of $\mathbf{X}$ for
numerical stability (Lecture 4, Section 3.3, Remark). We confirm the two routes agree
on the Wine training set by eigendecomposing $\mathbf{S}$ directly and comparing to
the SVD-derived components above.

In [ ]:
# Direct eigendecomposition of S, for comparison with the SVD-derived V and lambdas above
S_wine = (X_c.T @ X_c) / (n - 1)
eigvals_direct, eigvecs_direct = np.linalg.eigh(S_wine)   # ascending order
order = np.argsort(eigvals_direct)[::-1]
eigvals_direct = eigvals_direct[order]
eigvecs_direct = eigvecs_direct[:, order]

print("Eigenvalues from eigendecomposition of S:")
print(np.round(eigvals_direct, 4))
print("\nEigenvalues from SVD of X (lambdas, cell above):")
print(np.round(lambdas, 4))
print(f"\nMax absolute difference: {np.max(np.abs(eigvals_direct - lambdas)):.2e}")

# Eigenvectors can differ by a sign flip; compare absolute values column by column
sign_agreement = np.allclose(np.abs(eigvecs_direct), np.abs(V), atol=1e-6)
print(f"Eigenvectors agree up to sign: {sign_agreement}")

### C2. Variance explained and scree plot

In [ ]:
var_explained     = lambdas / lambdas.sum()
var_explained_cum = var_explained.cumsum()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].bar(range(1, len(lambdas)+1), var_explained * 100,
            color="steelblue", edgecolor="none", alpha=0.85)
axes[0].plot(range(1, len(lambdas)+1), var_explained_cum * 100,
             marker="o", color="firebrick", linewidth=1.5, markersize=4)
axes[0].axhline(90, color="grey", linestyle="--", linewidth=0.8)
axes[0].set_xlabel("Principal component")
axes[0].set_ylabel("Variance explained (%)")
axes[0].set_title("Scree plot — Wine dataset")
axes[0].set_xticks(range(1, len(lambdas)+1))

axes[1].plot(range(1, len(lambdas)+1), lambdas, marker="o",
             color="steelblue", linewidth=1.5, markersize=5)
axes[1].set_xlabel("Principal component")
axes[1].set_ylabel("Eigenvalue")
axes[1].set_title("Eigenvalues — Wine dataset")
axes[1].set_xticks(range(1, len(lambdas)+1))

plt.tight_layout()
plt.show()

print("Cumulative variance explained:")
for k, cv in enumerate(var_explained_cum, 1):
    print(f"  First {k:>2} component(s): {cv*100:.1f}%")

In [ ]:
# How many components to reach 90% variance explained?
q90 = np.searchsorted(var_explained_cum, 0.90) + 1
print(f"Components needed to explain >= 90% of variance: {q90} (out of {X_wine.shape[1]})")

### C3. Visualising the first two components

In [ ]:
Z2 = X_c @ V[:, :2]   # project onto first two PCs

class_names = wine.target_names
colors_cls  = ["steelblue", "darkorange", "forestgreen"]

fig, ax = plt.subplots(figsize=(7, 5))
for cls, name, color in zip([0, 1, 2], class_names, colors_cls):
    mask = y_wine_tr == cls
    ax.scatter(Z2[mask, 0], Z2[mask, 1], label=name, alpha=0.75,
               s=30, color=color, edgecolors="none")

ax.set_xlabel(f"PC1 ({var_explained[0]*100:.1f}% variance)")
ax.set_ylabel(f"PC2 ({var_explained[1]*100:.1f}% variance)")
ax.set_title("Wine dataset (training set) — first two principal components")
ax.legend()
plt.tight_layout()
plt.show()

### C4. Loadings: what do the components represent?

In [ ]:
# PC loadings: how much does each original feature contribute to each PC?
loadings = pd.DataFrame(
    V[:, :4],
    index=wine.feature_names,
    columns=[f"PC{k+1}" for k in range(4)]
)

print("Loadings (first 4 PCs):")
print(loadings.round(3).to_string())

# Visualise PC1 and PC2 loadings
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, pc in zip(axes, ["PC1", "PC2"]):
    vals = loadings[pc].sort_values()
    colors_bar = ["firebrick" if v < 0 else "steelblue" for v in vals]
    ax.barh(vals.index, vals.values, color=colors_bar, edgecolor="none", alpha=0.85)
    ax.axvline(0, color="black", linewidth=0.7)
    ax.set_title(f"{pc} loadings")
    ax.set_xlabel("Loading")
    ax.tick_params(labelsize=8)

plt.tight_layout()
plt.show()

**Exercise C.** Using only the first $q$ principal components where $q$ is the number
needed to explain at least 90% of the variance, fit a simple classifier (for example,
logistic regression) on the PCA-reduced Wine training set and evaluate it on the
held-out test set (`X_wine_te`, `y_wine_te`, projected using the same fitted scaler
and PC directions). Compare this to the same classifier fitted on all 13 original
(scaled) features. Then state one advantage and one disadvantage of working in the
reduced PCA space for this classification task.

---

## Part D — PCA inside a pipeline

PCA must obey the same train-only rule as all other preprocessing steps: the
principal component directions must be estimated on the training data and then
applied to the test data using those directions. A Pipeline enforces this.

### D1. PCA in sklearn

In [ ]:
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.metrics import root_mean_squared_error

# Fit PCA on the training set only
pca = PCA(n_components=0.95, random_state=0)  # retain components explaining 95% of variance
pca.fit(X_tr)

print(f"Components retained (95% variance): {pca.n_components_} out of {X_tr.shape[1]}")
print(f"Variance explained per component: {pca.explained_variance_ratio_.round(3)}")
print(f"Cumulative: {pca.explained_variance_ratio_.cumsum().round(3)}")

In [ ]:
# Full pipeline: scale -> PCA -> Ridge regression
pipe_pca = Pipeline([
    ("scaler", StandardScaler()),
    ("pca",    PCA(n_components=0.95, random_state=0)),
    ("model",  Ridge(alpha=1.0)),
])

pipe_baseline = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  Ridge(alpha=1.0)),
])

pipe_pca.fit(X_tr, y_tr)
pipe_baseline.fit(X_tr, y_tr)

rmse_pca      = root_mean_squared_error(y_te, pipe_pca.predict(X_te))
rmse_baseline = root_mean_squared_error(y_te, pipe_baseline.predict(X_te))

print(f"RMSE — baseline (all features):  {rmse_baseline:.4f}")
print(f"RMSE — PCA (95% variance):       {rmse_pca:.4f}")
print(f"\nComponents used by PCA pipeline: {pipe_pca.named_steps['pca'].n_components_}")

### D2. Variance explained vs RMSE

In [ ]:
# Sweep over the number of components and record test RMSE
n_features = X_tr.shape[1]
component_range = range(1, n_features + 1)
rmses = []

for q in component_range:
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("pca",    PCA(n_components=q, random_state=0)),
        ("model",  Ridge(alpha=1.0)),
    ])
    pipe.fit(X_tr, y_tr)
    rmses.append(root_mean_squared_error(y_te, pipe.predict(X_te)))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(component_range, rmses, marker="o", color="steelblue",
        linewidth=1.5, markersize=5)
ax.axhline(rmse_baseline, color="firebrick", linestyle="--",
           linewidth=1.2, label=f"Baseline (all {n_features} features)")
ax.set_xlabel("Number of principal components retained")
ax.set_ylabel("Test RMSE")
ax.set_title("California Housing — RMSE vs number of PCA components")
ax.legend()
plt.tight_layout()
plt.show()

best_q = int(np.argmin(rmses)) + 1
print(f"Best number of components: {best_q}  (RMSE = {min(rmses):.4f})")

The curve shows how much signal is captured by each additional component. For this
dataset the improvement flattens quickly, which is consistent with the scree plot
showing a small number of dominant directions. Notice that the best PCA model need
not outperform the baseline: PCA finds directions of maximum variance, not directions
most predictive of the target. When the predictive signal is spread across many
features, retaining all of them is better.

---

## Take-home exercises

**Exercise 1.** Load the Digits dataset (`sklearn.datasets.load_digits`, 64 features,
one per pixel of an 8x8 image). Scale it and compute its principal components via the
SVD, as in Part C. Plot the scree plot: how many components are needed to explain 90%
of the variance, and does the elbow look as clean as it did for Wine? Then pick a single
digit image, reconstruct it using only the first $q$ components for a few choices of
$q$ (e.g. 5, 20, 40, 64), and plot the reconstructions side by side with the original.
At what point does the reconstruction become recognisable? This is the same rank-$q$
truncation from the Eckart--Young theorem (Lecture 4, Exercise 2), applied visually
rather than numerically.

**Exercise 2.** Add the three engineered features from Part A (`bedroom_ratio`,
`rooms_per_person`, `income_per_occupant`) to the California Housing training set,
re-run PCA, and check whether the explained variance per component changes. Does
feature engineering before PCA affect how many components are needed?

**Exercise 3.** The Eckart--Young theorem (Exercise 2 in Lecture 4) states that the
rank-$q$ truncated SVD minimises the Frobenius approximation error. Verify this
numerically for the Wine training set: compute $\|\mathbf{X} - \hat{\mathbf{X}}_q\|_F^2$
for $q = 1, 2, \ldots, 13$ and check that it equals $\sum_{k=q+1}^{13} d_k^2$.

**Exercise 4.** (Written, no code.) PCA is an unsupervised method and does not use
the target variable. Describe a situation in which applying PCA before fitting a
classifier could hurt classification performance, even if PCA retains 99% of the
variance. What supervised alternative would be more appropriate in that case?

---